# 03 — Baseline Models


## Section 0 — Setup

Fail loudly at the top instead of quietly at cell 40. Three jobs: (a) prove which interpreter is executing, (b) make core/ importable regardless of where the kernel's cwd landed, (c) fix the seed and the MLflow experiment name once, so nothing downstream invents its own.

In [ ]:
import pandas as pd
import numpy as np
import scipy
from pathlib import Path
import sys 
import json

import sklearn
from sklearn.linear_model import Ridge
import xgboost
import optuna
import mlflow
import torch
import torch.nn as nn

print(sys.executable)
print(f'sklearn: {sklearn.__version__}\nxgboost: {xgboost.__version__}\noptuna: {optuna.__version__}\
    \nmlflow: {mlflow.__version__}\ntorch: {torch.__version__}\npandas: {pd.__version__}\nnumpy: {np.__version__}\nscipy: {scipy.__version__}')

CURRENT_FOLDER = Path.cwd()
ROOT_FOLDER = CURRENT_FOLDER.parent
assert (ROOT_FOLDER / 'main.py').exists(), f'not the repo root: {ROOT_FOLDER}'  # Asserting GripVT root

root_str = str(ROOT_FOLDER)
if root_str not in sys.path:
    sys.path.insert(0, root_str)

RANDOM_SEED = 42
EXPERIMENT_NAME = "gripvt-baseline-models"          # For MLflow


## Section 1 — Load Data

In [ ]:
RECORDINGS = ROOT_FOLDER / "recordings"

csv_paths = sorted(RECORDINGS.glob("participant*/session*/features_table.csv"))

sessions = []
for p in csv_paths:
    folder_name = p.parent.name
    session_id = int(folder_name.removeprefix('session').split('_')[0])
    sessions.append((session_id, p))

assert len(sessions) == 9, f"expected 9 sessions, got {len(sessions)}"
assert sorted(s_id for s_id, _ in sessions) == [1, 2, 3, 5, 6, 7, 8, 9, 10]

In [ ]:
df_list = []
calibration_modes = set()
git_commits = set()

for session_id, p in sessions:
    feature_table = pd.read_csv(p)                                  # Reads features_table.csv for each path in sessions\

    participant_id = str(p.parent.parent.name.split('_')[0])        # Gets participant number
    feature_table['participant_id'] = participant_id
    feature_table['session_id'] = session_id

    df_list.append(feature_table)

    with open(p.parent / 'features_meta.json', encoding='utf-8') as f:
        meta = json.load(f)                                         # something to remember: load for file, loads for string

    calibration_modes.add(meta['calibration_mode'])

    git_commits.add(meta['git_commit'])

df = pd.concat(df_list, ignore_index=True)

assert df.shape == (28260, 808), f"expected (28260, 808), got {df.shape}"
assert df.index.is_unique, f"expected True, got {df.index.is_unique}"
assert df.index[-1] == 28259, f"expected True, got {df.index[-1]}"
assert pd.api.types.is_integer_dtype(df.session_id), f"expected int,  got {df.session_id.dtype}"
assert calibration_modes == {"per_wear"},  f"mixed calibration: {calibration_modes}"
assert git_commits == {"0018b43"}, f"mixed git commits: {git_commits}"

df.groupby("session_id").size()

## Section 2 — Read Folds

In [ ]:
with open(ROOT_FOLDER / 'notebooks' / 'split_manifest.json', encoding='utf-8') as f:
    manifest = json.load(f)

session_folds = manifest['session_folds']
loaded_sessions = set(df['session_id'].unique())

assert len(session_folds) == 9, f"len(session_folds) expected 9, got {len(session_folds)}"

for fold in session_folds:
    train_sessions = fold['train_sessions']
    test_session = fold['test_session']

    assert {test_session} | set(train_sessions) == loaded_sessions, \
        f"fold {fold['fold']}: sessions {sorted({test_session} | set(train_sessions))} != loaded {sorted(loaded_sessions)}"
    assert test_session not in train_sessions, \
        f"fold {fold['fold']}: test session {test_session} leaks into train_sessions"
    assert len(train_sessions) == 8, \
        f"fold {fold['fold']}: len(train_sessions) expected 8, got {len(train_sessions)}"

test_sessions = sorted(fold['test_session'] for fold in session_folds)
assert test_sessions == sorted(loaded_sessions), \
    f"each session must be held out exactly once, got {test_sessions}"

assert manifest['random_seed'] == RANDOM_SEED, \
    f"expected manifest random_seed {RANDOM_SEED}, got {manifest['random_seed']}"

## Section 3 — Dropping velocity-NaN rows

In [ ]:
df.isna().any(axis=1).sum()             # Checking how many rows have NaN values

In [ ]:
df = df.dropna(subset=df.filter(regex='vx|vy').columns)

assert df.shape[0] == 27991, f"expected row count is 27,991, got {df.shape[0]}"
assert df.isna().any().sum() == 0, f"NaN rows: {df[df.isna().any(axis=1)]}"

# A 1-frame trial would vanish entirely in the drop above, shrinking a fold silently.
# A trial is the pair (session_id, trial_id) -- trial_id alone runs 0-29 in every session.
n_trials = df.groupby(['session_id', 'trial_id']).ngroups
assert n_trials == 269, f"expected 269 trials after the drop, got {n_trials}"
assert df.shape[1] == 808, f"expected 808 columns, got {df.shape[1]}"

## Section 4 — Feature Matrix and Channel Groups

In [ ]:
# Getting unique columns prefixes (i.e. vx, vy, etc) 
unique_prefix = df.columns.str.replace(r'[\d_]', '', regex=True).unique()
print(unique_prefix)

# Remove non_feature rows
unwanted_prefixes = unique_prefix[~unique_prefix.isin(['dx', 'dy', 'dA', 'mag', 'strainx', 
                                                       'strainy', 'vx', 'vy', 'maxdisp'])].tolist()
print(unwanted_prefixes)

In [ ]:
unique_prefix = df.columns.str.replace(r'\d+$', '', regex=True).unique()        # Remove only the trailing numbers from the column names using r'\d+$'

print(f"Extracted Unique Prefixes: {unique_prefix}")

allowed_prefixes = ['dx_', 'dy_', 'dA_', 'mag_', 'strain_x_', 'strain_y_', 'vx_', 'vy_', 'max_disp']

unwanted_prefixes = unique_prefix[~pd.Index(unique_prefix).isin(allowed_prefixes)].tolist() # Filter out the allowed prefixes to find the unwanted ones

print(f"Unwanted Prefixes Identified: {unwanted_prefixes}\nCount: {len(unwanted_prefixes)}")

In [ ]:
# Store all feature columns in an ordered list
allowed_tuple = tuple(allowed_prefixes)         # Convert to tuple, required for str.startswith()
matching_cols = df.columns[df.columns.str.startswith(allowed_tuple)].tolist()

feature_cols = matching_cols
assert len(feature_cols) == 793, f"expected 793, got {len(feature_cols)}" 

TARGET = 'pct_mvc_corr'
assert TARGET not in feature_cols, f"pct_mvc_corr in feature_cols"

print(feature_cols)

In [ ]:
channel_groups = {
    "dx_dy": [c for c in feature_cols if 'dx' in c or 'dy' in c],
    "dA": [c for c in feature_cols if 'dA' in c],
    "mag": [c for c in feature_cols if 'mag' in c],
    "strain": [c for c in feature_cols if 'strain_x' in c or 'strain_y' in c],
    "velocity": [c for c in feature_cols if 'vx' in c or 'vy' in c],
    "max_disp": ['max_disp']
}

grouped_cols = [c for cols in channel_groups.values() for c in cols]

assert len(channel_groups) == 6, \
    f"expected 6 groups, got {len(channel_groups)}: {list(channel_groups.keys())}"

# Disjoint: a column claimed by two groups shows up twice in grouped_cols.
assert len(grouped_cols) == len(set(grouped_cols)), \
    f"groups overlap: {sorted({c for c in grouped_cols if grouped_cols.count(c) > 1})}"

# Covering: nothing left out, nothing invented. Disjoint + covering = a partition.
assert set(grouped_cols) == set(feature_cols), \
    f"missing: {sorted(set(feature_cols) - set(grouped_cols))}, extra: {sorted(set(grouped_cols) - set(feature_cols))}"

assert len(grouped_cols) == 793, f"expected 793 grouped columns, got {len(grouped_cols)}"

print({name: len(cols) for name, cols in channel_groups.items()})

## Section 5 — Metrics

One function used by every model, rung and ablation variant. Returns 9 rows: overall,
3 phase slices, 5 level slices. `braced` is a subset of `pre_cue`, so the phase slices
overlap — only the level slices partition the rows.

RMSE in %MVC points is the headline (directly comparable to EchoForce's 10.10 %).
R² is scored against each slice's own variance, so it goes sharply negative on narrow
slices even when RMSE is fine. That is arithmetic, not a broken model.

**Predict-the-mean baseline over all 27,991 rows — the floor any model must beat:**

| slice | n | mean_y | RMSE | MAE | R² |
|---|---|---|---|---|---|
| all | 27,991 | 7.40 | **15.03** | 9.86 | 0.000 |
| pre_cue | 13,436 | 1.32 | 7.12 | 6.68 | −2.669 |
| post_cue | 14,555 | 13.00 | 19.69 | 12.79 | −0.088 |
| braced | 4,365 | 1.16 | 6.33 | 6.24 | −33.677 |
| level 0 | 4,548 | 1.27 | 7.24 | 6.90 | −2.537 |
| level 5 | 5,247 | 2.08 | 5.70 | 5.34 | −6.693 |
| level 20 | 5,979 | 6.03 | 8.64 | 7.56 | −0.026 |
| level 50 | 5,939 | 11.32 | 17.55 | 12.65 | −0.053 |
| level 80 | 6,278 | 13.88 | 24.09 | 15.32 | −0.078 |

R² is exactly 0.000 on the `all` row by definition — that is the check that proves the
function is wired correctly. Level 80 averages only 13.88 %MVC, consistent with the
participant001 finding that 80 % MVC was never actually reached.

In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

def evaluate(y_true, y_pred, meta):
    ''' One row per reporting slice: Overall, By Phase, By Level
        y_true  : 1-D array or Series of true %MVC
        y_pred  : 1-D numpy array, same length and same order
        meta    : DataFrame with 'phase', 'braced', 'target_pct', same length and order

        'braced' is a subset of 'pre_cue', so the phase slices overlap and their n do
        not sum to len(y_true). The five level slices do partition the rows.
    '''
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Masks go to numpy so every lookup below is positional, never index-aligned.
    slices = [("all", "all", np.ones(len(y_true), dtype=bool))]
    slices += [("phase", p, (meta["phase"] == p).to_numpy()) for p in ("pre_cue", "post_cue")]
    slices += [("phase", "braced", meta["braced"].to_numpy())]
    slices += [("level", lvl, (meta["target_pct"] == lvl).to_numpy()) for lvl in (0, 5, 20, 50, 80)]

    rows = []
    for slice_type, slice_name, mask in slices:
        yt, yp = y_true[mask], y_pred[mask]
        rows.append({
            "slice_type": slice_type,
            "slice": slice_name,
            "n": len(yt),
            "mean_y": np.mean(yt),
            "rmse": root_mean_squared_error(yt, yp),
            "mae": mean_absolute_error(yt, yp),
            "r2": r2_score(yt, yp),
        })

    return pd.DataFrame(rows)

In [ ]:
# Sanity-check evaluate() before any model exists.
meta_all = df[["phase", "braced", "target_pct"]]
y_all = df[TARGET]

perfect = evaluate(y_all, y_all.to_numpy(), meta_all)                       # predict the truth
constant = evaluate(y_all, np.full(len(y_all), y_all.mean()), meta_all)     # predict the mean

assert len(perfect) == 9, f"expected 9 slices, got {len(perfect)}"
assert (perfect["rmse"] == 0).all(), "perfect prediction must give RMSE 0 everywhere"
assert (perfect["r2"] == 1).all(), "perfect prediction must give R2 1 everywhere"
assert (constant["rmse"] >= constant["mae"]).all(), "RMSE < MAE is impossible"

# pre_cue + post_cue covers every row; braced is a subset of pre_cue and is left out.
phase_n = perfect.loc[perfect["slice"].isin(["pre_cue", "post_cue"]), "n"].sum()
level_n = perfect.loc[perfect["slice_type"] == "level", "n"].sum()
assert phase_n == len(df), f"phase slices cover {phase_n}, expected {len(df)}"
assert level_n == len(df), f"level slices cover {level_n}, expected {len(df)}"

constant.round(3)

## 6 — Nested Tuning Loop

In [ ]:
"""
    Take one fold dictionary -> pull out train_sessions and test_session
    ASSERT: test_session not in train_session
    Loop 8 times.  On loop i, one of the 8 training sessions becomes the inner validation session, and the other 7 become the inner training set.
    Each time: build fresh model from params, fit on 7, predict the 1, compute RMSE
    Return mean of 8 RMSE numbers, float
"""

def inner_cv_score(fold, params, make_model):
    ''' Mean RMSE over one outer fold's 8 inner leave-one-session-out splits.
        fold        : ONE dict from session_folds
        params      : hyperparameter dict, handed straight to make_model
        make_model  : callable(params) -> a fresh object with .fit / .predict

        The outer test session never appears here: inner splits are drawn only from this
        fold's 8 training sessions. Returns one float so Optuna has something to minimise.
        Plain aggregate RMSE, not evaluate() -- tuning needs one number, not 9 rows.
    '''
    train_sessions = fold['train_sessions']
    assert fold['test_session'] not in train_sessions, \
        f"fold {fold['fold']}: test session {fold['test_session']} leaks into {train_sessions}"

    scores = []
    for val_session in train_sessions:
        inner_train = [s for s in train_sessions if s != val_session]    # derive the 7, never mutate the 8

        train_mask = df['session_id'].isin(inner_train)                  # masks rebuilt every pass
        val_mask = df['session_id'] == val_session

        model = make_model(params)                                       # fresh model per split, no warm start
        model.fit(df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET])
        y_pred = model.predict(df.loc[val_mask, feature_cols])

        scores.append(root_mean_squared_error(df.loc[val_mask, TARGET], y_pred))

    return float(np.mean(scores))

In [ ]:
# Smoke test: no Optuna yet. One fold, one fixed alpha, 8 inner fits -> one number and a timing.
# StandardScaler lives inside a Pipeline so it is fitted on the 7 training sessions only,
# never on all of df. That is the defence against the scaler leak.
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

t0 = time.perf_counter()
smoke_score = inner_cv_score(
    session_folds[0],
    {},
    lambda p: Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.0))]),
)
print(f"fold 0 (test session {session_folds[0]['test_session']}) mean inner RMSE: {smoke_score:.3f} %MVC")
print(f"{time.perf_counter() - t0:.1f} s for 8 fits")

In [ ]:
# Two checks on inner_cv_score, both instant.
fold0 = session_folds[0]

# 1. The 8 training sessions hold 24,354 rows: 27,991 minus session 1's 3,637.
assert df['session_id'].isin(fold0['train_sessions']).sum() == 24354

# 2. The leak guard must actually fire. make_model is None because the assert raises
#    before any model is built. Copy the fold -- never mutate the manifest's list.
bad_fold = {**fold0, 'train_sessions': fold0['train_sessions'] + [fold0['test_session']]}
guard_fired = False
try:
    inner_cv_score(bad_fold, {}, None)
except AssertionError:
    guard_fired = True
assert guard_fired, "leak guard did not fire on a fold with the test session in train_sessions"

print("inner_cv_score: 24,354 training rows, leak guard fires")

In [ ]:
import time

# mlflow 3.15 REFUSES the legacy ./mlruns file store -- it raises "maintenance mode" unless
# you opt out with an env var. Point it at sqlite instead, which is what mlflow recommends.
# Read the results with:  mlflow ui --backend-store-uri sqlite:///mlflow.db
MLFLOW_DB = f"sqlite:///{(ROOT_FOLDER / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment(EXPERIMENT_NAME)

# Budget -- the SAME MAX_TRIALS and PATIENCE for every model. That is the fairness contract.
MAX_TRIALS = 100
PATIENCE = 15

optuna.logging.set_verbosity(optuna.logging.WARNING)


def stop_when_flat(study, trial):
    ''' Study-level early stopping -- the budget rule. Optuna ships no patience stopper, but
        the study already records which trial holds the best value, so "trials since the last
        improvement" needs no counter. Pruning stays OFF: this is the only stopping rule.
    '''
    if trial.number - study.best_trial.number >= PATIENCE:
        study.stop()

## Section 7 — Models

One cell per model, each holding its own flat 9-fold loop. The three cells are deliberate
copy-paste siblings: same folds, same `MAX_TRIALS` / `PATIENCE`, same `inner_cv_score`, same
`evaluate()`. Only the params dict and the model constructor differ — that is the fairness
contract, and it is what makes "XGBoost beat Ridge" a statement about models rather than
about scoring code.

### 7a - Ridge Regression

In [ ]:
# 7a -- Ridge.
# StandardScaler must live INSIDE the Pipeline: the Pipeline fits it on exactly the rows handed
# to .fit(), so it never sees the inner validation session or the outer test session.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_ridge(params):
    return Pipeline([("scale", StandardScaler()), ("ridge", Ridge(**params))])  # Data is normalized here

ridge_rows = []

# 1. Search, using 8 training sessions only
for fold in session_folds:
    def objective(trial):
        params = {
            "alpha": trial.suggest_float("alpha", 1e-3, 1e6, log=True)
        }
        return inner_cv_score(fold, params, make_ridge)

    # Run Optuna
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=MAX_TRIALS, callbacks=[stop_when_flat])   # FIXED n_trial -> n_trials

    # 2. Refit on all 8, then touch the test session exactly once
    # FIXED: these were copied from inner_cv_score, so they referenced inner_train and
    # val_session -- names that only exist inside that function. Out here the split is the
    # OUTER one: all 8 training sessions vs the single held-out session.
    train_mask = df['session_id'].isin(fold['train_sessions'])
    test_mask = df['session_id'] == fold['test_session']

    model = make_ridge(study.best_params)
    t0 = time.perf_counter()
    model.fit(df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET])
    fit_seconds = time.perf_counter() - t0

    y_pred = model.predict(df.loc[test_mask, feature_cols])
    y_true = df.loc[test_mask, TARGET]
    table = evaluate(y_true, y_pred, df.loc[test_mask, ['phase', 'braced', 'target_pct']])

    # 3. Label 9 rows so they survive concat
    table['model'] = 'ridge'
    table['fold'] = fold['fold']
    table['test_session'] = fold['test_session']
    table['n_trials'] = len(study.trials)
    table['fit_seconds'] = fit_seconds

    # 4. One MLflow run per (model, fold)
    # FIXED: log_metrics takes ONE dict of numbers, not a name and a value. The 'all' row also
    # carries the string columns slice_type/slice, which MLflow rejects, so name the numbers.
    overall = table.loc[table['slice'] == 'all'].iloc[0]

    with mlflow.start_run(run_name=f"ridge-fold{fold['fold']}"):
        mlflow.log_params(study.best_params)
        mlflow.log_metrics({
            "test_rmse": overall['rmse'],
            "test_mae": overall['mae'],
            "test_r2": overall['r2'],
            "inner_rmse_best": study.best_value,
            "n_trials": len(study.trials),
            "fit_seconds": fit_seconds,
        })

    # FIXED: print the real numbers -- nbstripout erases outputs, so this is the only live
    # feedback during the run, and the achieved trial count is itself a reported result.
    print(f"fold {fold['fold']} test{fold['test_session']}: RMSE {overall['rmse']:.3f} | "
          f"inner {study.best_value:.3f} | {len(study.trials)} trials | "
          f"alpha {study.best_params['alpha']:.4g}")

    ridge_rows.append(table)

ridge_results = pd.concat(ridge_rows, ignore_index=True)

In [ ]:
ridge_results.loc[ridge_results['slice'] == 'all', 'n'].sum()          # must be 27991

In [ ]:
# by-slice .mean() of n/rmse/mae/r2
ridge_results.groupby(['slice_type','slice'], sort=False)[['n','rmse','mae','r2']].mean().round(3)

In [ ]:
# per-fold all rows
ridge_results.loc[ridge_results['slice'] == 'all', ['test_session','rmse','mae','r2','n_trials']].round(3)

In [ ]:
# by-slice rmse.agg(['mean','std','min','max']) — the spread
ridge_results.groupby(['slice_type','slice'], sort=False)['rmse'].agg(['mean','std','min','max']).round(3)

In [ ]:
study.trials_dataframe()['value'].describe()

### 7a results — Ridge, rung 1 (cross-mount, cold)

Run 2026-08-11 · 44 min · `MAX_TRIALS = 100`, `PATIENCE = 15`, seed 42.

Nine folds. Each one trains on eight sessions and is tested on the ninth, so every fold
answers: *how well does this work on a wearing of the sleeve it has never seen?*

#### Per fold

| fold | test session | test RMSE | inner RMSE | trials | alpha |
|---|---|---|---|---|---|
| 0 | 1 | 6.105 | 6.808 | 45 | 2.446e+04 |
| 1 | 2 | 7.460 | 6.752 | 97 | 3.137e+04 |
| 2 | 3 | 5.259 | 7.084 | **100 (cap)** | 4.384e+04 |
| 3 | 5 | 5.974 | 6.908 | 38 | 4.175e+04 |
| 4 | 6 | 7.318 | 6.743 | 38 | 4.175e+04 |
| 5 | 7 | **13.216** | 6.069 | 31 | 2.485e+04 |
| 6 | 8 | 8.219 | 6.901 | 47 | 5.136e+04 |
| 7 | 9 | 4.619 | 7.206 | 47 | 5.136e+04 |
| 8 | 10 | 4.897 | 7.202 | 47 | 5.136e+04 |

**Mean 7.007 %MVC, SD 2.632, range 4.62 – 13.22.** Predicting the training mean scores 15.03,
so Ridge is well clear of the floor.

Two things to say carefully:

- This is the **average of nine per-fold RMSEs**. Pooling all 27,991 predictions into one RMSE
  instead gives **7.295**. The average-of-folds is the one to quote — it is what SO4 asks for
  and it is the same quantity as EchoForce's 10.10 % — but say which one you mean.
- **Session 7 (13.216) is not excluded.** It is 9.7 % of the rows but 31.9 % of the total
  squared error. Dropping it would be picking the data after seeing the score. Without it the
  mean is 6.231 (SD 1.312). Report both; exclude neither. The session that *was* excluded is
  **session 4**, on an a-priori rule — its force export contains zero data rows, so it has no
  labels to train against.

#### What the phase slices mean

Every trial has a moment where the operator says "go" — the `grip_start` event. `to_cue` is the
number of seconds from that moment: negative before, positive after. The three slices are just
windows on that clock.

| slice | when | what the participant is doing |
|---|---|---|
| `pre_cue` | before "go" | waiting, at rest — around 1 %MVC no matter what the trial is labelled |
| `post_cue` | after "go" | actually squeezing |
| `braced` | the last 1 s before "go" | tensing up in anticipation. This is **part of `pre_cue`**, not a fourth phase |

Because `braced` sits inside `pre_cue`, the three row counts do not add up to the total. Only
the five target levels partition the rows.

#### By phase and by level

| slice | rows | RMSE | MAE | R² | floor |
|---|---|---|---|---|---|
| **all** | 27,991 | **7.007** | 4.492 | 0.755 | 15.03 |
| `pre_cue` | 13,436 | 4.048 | 2.922 | −0.661 | 7.12 |
| **`post_cue`** | 14,555 | **8.764** | 5.874 | 0.752 | 19.69 |
| `braced` | 4,365 | 4.070 | 3.150 | −21.399 | 6.33 |
| level 0 | 4,548 | 3.590 | 2.716 | −54.015 | 7.24 |
| level 5 | 5,247 | 3.158 | 2.640 | −1.903 | 5.70 |
| level 20 | 5,979 | 5.831 | 4.329 | 0.436 | 8.64 |
| level 50 | 5,939 | 7.868 | 5.448 | 0.752 | 17.55 |
| level 80 | 6,278 | **10.347** | 6.652 | 0.778 | 24.09 |

**The number to quote is post-cue 8.764, not the aggregate 7.007.**

About half of every row in the dataset is pre-cue rest sitting near zero force. Predicting near
zero when the truth is near zero is easy, and the model does it well (4.048). So the aggregate
is an average of an easy interval and a hard one — it credits the model for the half of the
recording where there is nothing to measure. The sensor's job is to read force *while someone
is gripping*, and that is the post-cue interval alone. Post-cue 8.764 is still under EchoForce's
10.10 %.

The same effect shows up in the level labels: the "80 %" trials average only 13.9 %MVC across
their full duration, because half of each one is rest. **The levels are instructed targets, not
achieved force.**

Three notes on reading the table:

- `braced` (4.070) is essentially identical to `pre_cue` (4.048), so anticipatory tension does
  not appear to confuse the model.
- **Negative R² on `braced` and level 0 is arithmetic, not failure.** R² compares the model
  against that slice's own variance, and those slices are nearly constant, so the denominator
  collapses. RMSE is the headline for exactly this reason. These are also averages of nine
  per-fold R², which is unbounded below — one bad fold can drag level 0 to −54.
- Every slice's row count matches the §5 floor table exactly, which proves each row was
  predicted **exactly once** across the nine folds. No fold dropped or double-counted rows.

#### Cost, and one caveat about the search

490 trials total, mean 54 per fold, **1 of 9 hit the 100-trial cap.** About 0.7 s per Ridge fit,
roughly 5,100 fits.

⚠ **The repeated alphas are not nine independent searches agreeing.** Folds 6–8 report the same
alpha *and* the same trial count, as do folds 3–4. `TPESampler` draws its first ten trials from
the seeded RNG, so every fold explores the same first ten alphas before the search adapts
(verified: two studies with unrelated objectives gave identical trials 0–9). The defensible
claim is *"the tuned penalty was consistently large, 2.4–5.1×10⁴"* — nothing stronger.

The objective is also nearly flat near its optimum. Fold 8's 47 trials ranged 7.202 – 9.670,
with a quarter of them within 0.083 of the best. That is why the patience rule often fails to
fire and studies ride the cap — it counts any improvement at all, including 1e-9.

### 7b - XGBoost

In [ ]:
from xgboost import XGBRegressor

fold = session_folds[0]
train_mask = df['session_id'].isin(fold['train_sessions'])

xgb = XGBRegressor(device="cuda", tree_method="hist", n_estimators=300, random_state=RANDOM_SEED)

t0 = time.perf_counter()
xgb.fit(df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET])
print(f"{time.perf_counter() - t0:.2f} s for one fit on {train_mask.sum():,} rows x {len(feature_cols)} features")